In [1]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import open3d as o3d
from plotly.subplots import make_subplots

class HeadlessPLYViewer:
    def __init__(self):
        self.mesh = None
        self.vertices = None
        self.faces = None
        self.colors = None
    
    def load_ply(self, file_path):
        """Load PLY file for headless viewing"""
        self.mesh = o3d.io.read_triangle_mesh(file_path)
        
        if len(self.mesh.vertices) == 0:
            print("Failed to load mesh")
            return False
        
        # Extract data for plotting
        self.vertices = np.asarray(self.mesh.vertices)
        self.faces = np.asarray(self.mesh.triangles)
        
        if self.mesh.has_vertex_colors():
            self.colors = np.asarray(self.mesh.vertex_colors)
        else:
            self.colors = None
            
        print(f"Loaded mesh: {len(self.vertices)} vertices, {len(self.faces)} faces")
        return True
    
    def create_interactive_plotly(self, show_wireframe=False, point_size=2):
        """Create interactive Plotly 3D visualization"""
        if self.vertices is None:
            print("No mesh loaded")
            return None
        
        fig = go.Figure()
        
        # Prepare color data
        if self.colors is not None:
            # Convert colors to RGB strings
            color_rgb = (self.colors * 255).astype(int)
            vertex_colors = [f'rgb({r},{g},{b})' for r, g, b in color_rgb]
        else:
            vertex_colors = 'blue'
        
        # Add mesh
        if len(self.faces) > 0:
            fig.add_trace(go.Mesh3d(
                x=self.vertices[:, 0],
                y=self.vertices[:, 1],
                z=self.vertices[:, 2],
                i=self.faces[:, 0],
                j=self.faces[:, 1],
                k=self.faces[:, 2],
                vertexcolor=vertex_colors if self.colors is not None else None,
                opacity=0.8,
                name="Mesh",
                showscale=False
            ))
        
        # Add wireframe if requested
        if show_wireframe:
            edge_trace = self._create_wireframe()
            if edge_trace:
                fig.add_trace(edge_trace)
        
        # Add point cloud view
        fig.add_trace(go.Scatter3d(
            x=self.vertices[:, 0],
            y=self.vertices[:, 1],
            z=self.vertices[:, 2],
            mode='markers',
            marker=dict(
                size=point_size,
                color=vertex_colors if self.colors is not None else 'red',
                opacity=0.6
            ),
            name="Points",
            visible=False  # Hidden by default
        ))
        
        # Update layout
        fig.update_layout(
            title="Interactive 3D Mesh Viewer",
            scene=dict(
                xaxis_title="X",
                yaxis_title="Y",
                zaxis_title="Z",
                camera=dict(
                    eye=dict(x=1.5, y=1.5, z=1.5)
                ),
                aspectmode='data'
            ),
            width=900,
            height=700,
            updatemenus=[
                dict(
                    type="buttons",
                    direction="left",
                    buttons=list([
                        dict(
                            args=[{"visible": [True, show_wireframe, False]}],
                            label="Mesh",
                            method="restyle"
                        ),
                        dict(
                            args=[{"visible": [False, False, True]}],
                            label="Points",
                            method="restyle"
                        ),
                        dict(
                            args=[{"visible": [True, show_wireframe, True]}],
                            label="Both",
                            method="restyle"
                        )
                    ]),
                    pad={"r": 10, "t": 10},
                    showactive=True,
                    x=0.01,
                    xanchor="left",
                    y=1.02,
                    yanchor="top"
                ),
            ]
        )
        
        return fig
    
    def _create_wireframe(self):
        """Create wireframe edges for the mesh"""
        if self.faces is None or len(self.faces) == 0:
            return None
            
        edges = set()
        for face in self.faces:
            edges.add(tuple(sorted([face[0], face[1]])))
            edges.add(tuple(sorted([face[1], face[2]])))
            edges.add(tuple(sorted([face[2], face[0]])))
        
        edge_x, edge_y, edge_z = [], [], []
        for edge in edges:
            x0, y0, z0 = self.vertices[edge[0]]
            x1, y1, z1 = self.vertices[edge[1]]
            edge_x.extend([x0, x1, None])
            edge_y.extend([y0, y1, None])
            edge_z.extend([z0, z1, None])
        
        return go.Scatter3d(
            x=edge_x, y=edge_y, z=edge_z,
            mode='lines',
            line=dict(color='black', width=1),
            name="Wireframe",
            showlegend=False
        )
    
    def show_mesh_comparison(self, file_paths, titles=None):
        """Compare multiple PLY files side by side"""
        n_meshes = len(file_paths)
        if n_meshes > 4:
            print("Maximum 4 meshes supported for comparison")
            return
        
        if n_meshes == 1:
            cols, rows = 1, 1
        elif n_meshes == 2:
            cols, rows = 2, 1
        else:
            cols, rows = 2, 2
        
        fig = make_subplots(
            rows=rows, cols=cols,
            specs=[[{'type': 'scene'}] * cols for _ in range(rows)],
            subplot_titles=titles or [f"Mesh {i+1}" for i in range(n_meshes)]
        )
        
        for i, file_path in enumerate(file_paths):
            if self.load_ply(file_path):
                row = i // cols + 1
                col = i % cols + 1
                
                # Prepare colors
                if self.colors is not None:
                    color_rgb = (self.colors * 255).astype(int)
                    vertex_colors = [f'rgb({r},{g},{b})' for r, g, b in color_rgb]
                else:
                    vertex_colors = px.colors.qualitative.Set1[i % len(px.colors.qualitative.Set1)]
                
                # Add mesh
                fig.add_trace(
                    go.Mesh3d(
                        x=self.vertices[:, 0],
                        y=self.vertices[:, 1],
                        z=self.vertices[:, 2],
                        i=self.faces[:, 0],
                        j=self.faces[:, 1],
                        k=self.faces[:, 2],
                        vertexcolor=vertex_colors,
                        opacity=0.8,
                        showscale=False
                    ),
                    row=row, col=col
                )
        
        fig.update_layout(height=800, title_text="Mesh Comparison")
        return fig

# Usage example
viewer = HeadlessPLYViewer()

# Single mesh viewing
ply_file = "/blue/prabhat/duminduaelamurem/wd/repo_tests/aaai/mvdust3r/rendering_results/mesh/01-123456.ply"
if viewer.load_ply(ply_file):
    fig = viewer.create_interactive_plotly(show_wireframe=True)
    fig.show()

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Open3D WARNING] Read PLY failed: unable to open file: /blue/prabhat/duminduaelamurem/wd/repo_tests/aaai/mvdust3r/rendering_results/mesh/01-123456.ply
Failed to load mesh


RPly: Unable to open file
